In [2]:
from proj_utils import *

import os
import pickle
from os.path import join
from copy import deepcopy
from collections import defaultdict, Counter
from itertools import combinations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import ipywidgets as widgets
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm

import sklearn as sk
from scipy.optimize import linear_sum_assignment
from scipy.optimize import curve_fit
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

import bertopic
from sentence_transformers import SentenceTransformer

from proj_utils import *

/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/pyth

### Caution : before run this, run [transform] in 'fit-final-models.py' and time-analysis.py.

In [3]:
collection_name = 'Breitbart'
model_name = MODEL_NAMES[collection_name]
threshold = 10

start_date, end_date = DATE_RANGES[collection_name]
num_topics = NUM_TOPICS[collection_name]
topic_model = (BERTopic.load(join('model', collection_name.lower(), model_name), embedding_model="all-MiniLM-L6-v2"))

In [ ]:
# open pickle files
with open(join('article', collection_name.lower(), MODEL_NAMES[collection_name], 'article_dict.pkl'), 'rb') as f:
    article_dict = pickle.load(f)

In [ ]:
# load articles_by_day from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'rb') as f:
    articles_by_day = pickle.load(f)
    
# remove articles with less than threshold comments
for day, articles in articles_by_day.items():
    articles_by_day[day] = articles[articles['comment_id'].apply(len) > threshold]


In [ ]:
date_cutoff = 7

for article_day in articles_by_day.keys():
    print(article_day)
    articles = articles_by_day[article_day]
    comment_createdAt_list = []
    comment_id_list = []
    comment_topics_list = []
    comments_embeddings_list = []
    for article in articles.itertuples():
        filtered_index = np.where(np.array([(comment_createdAt - article.createdAt).days for comment_createdAt in article.comment_createdAt]) < date_cutoff)[0]
        comment_createdAt_list.append([article.comment_createdAt[i] for i in filtered_index])
        comment_id_list.append([article.comment_id[i] for i in filtered_index])
        comment_topics_list.append([article.comment_topics[i] for i in filtered_index])
        comments_embeddings_list.append(np.array([article.comments_embeddings[i] for i in filtered_index]))

    articles_by_day[article_day] = articles_by_day[article_day].assign(comment_id=comment_id_list, comment_topics=comment_topics_list, comment_createdAt=comment_createdAt_list, comments_embeddings=comments_embeddings_list)

In [ ]:
# merge articles_by_day into articles_by_'month' (not month, but given time interval)

aggregate_days = 7

articles_by_month = dict()
keys = sorted(articles_by_day.keys(), key=lambda x: datetime.strptime(x, '%Y-%m-%d'))

start = datetime.strptime(keys[0], '%Y-%m-%d')
index = 0
while True:
    end = start + relativedelta(days=aggregate_days)
    print(start, end)
    if end > datetime.strptime(keys[-1], '%Y-%m-%d'):
        break
    tmp_list = []
    while True:
        if datetime.strptime(keys[index], '%Y-%m-%d') >= end:
            break
        else:
            tmp_list.append(articles_by_day[keys[index]])
            del articles_by_day[keys[index]]  # for the memory
            index += 1
    if len(tmp_list) > 0:
        articles_by_month[start.strftime('%Y-%m-%d')] = pd.concat(tmp_list)
    start = end

## 1. articles_by_day from article_dict

In [ ]:
# from pymongo collection, query articles with given id
def get_articles_from_ids(collection, ids, select_columns):
    query = {'_id': {'$in': ids}}
    projection = {k: 1 for k in select_columns}
    articles = collection.find(query, projection)
    return pd.DataFrame(articles)

# from pymongo collection, query articles with given date range
def get_articles_from_date(collection, start_date, end_date, select_columns):
    query = {'createdAt': {'$gte': start_date, '$lt': end_date}}
    projection = {k: 1 for k in select_columns}
    articles = collection.find(query, projection)
    return pd.DataFrame(articles)

# get comments with given article id
def get_comments_from_id(collection, id, select_columns):
    query = {'art_id': id}
    projection = {k: 1 for k in select_columns}
    comments = collection.find(query, projection)
    return pd.DataFrame(comments)

In [ ]:
def get_articles_by_day(collection_articles, collection_comments, start_date, end_date, select_columns):
    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    articles_by_day = defaultdict(list)
    while start_date < end_date:
        next_day = start_date + relativedelta(days=1)
        articles = get_articles_from_date(collection_articles, start_date, next_day, select_columns)
        if len(articles) > 0:
            articles['title_embeddings'] = sentence_model.encode(articles['clean_title'].values.tolist(), show_progress_bar=True, convert_to_tensor=True).tolist()
        articles_by_day[start_date.strftime("%Y-%m-%d")] = articles
        start_date = next_day
    return articles_by_day

In [ ]:
from proj_utils import _init_mongo_collection

select_columns = ['_id', 'clean_title', 'createdAt']
mongo_client_articles, collection_articles = _init_mongo_collection('Articles', collection_name)
mongo_client_comments, collection_comments = _init_mongo_collection('Comments', collection_name)

In [ ]:
articles_by_day = get_articles_by_day(collection_articles, collection_comments, start_date, end_date, select_columns)

In [ ]:
# save articles_by_day to pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
    pickle.dump(articles_by_day, f)

In [ ]:
# load articles_by_day from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'rb') as f:
    articles_by_day = pickle.load(f)

In [ ]:
# pop element from articles_by_day if there are no articles
for day in list(articles_by_day.keys()):
    if len(articles_by_day[day]) == 0:
        articles_by_day.pop(day)

current_date = start_date
while current_date < end_date:
    print(current_date)
    class_nums = []
    class_probs = []
    articles = articles_by_day[current_date.strftime("%Y-%m-%d")]
    if len(articles):
        if 'topic_num' in articles.columns:
            print('Already processed', current_date.strftime("%Y-%m-%d"))
            current_date += relativedelta(days=1)
            pass
        else:
            for i in range(len(articles)):
                output = topic_model.find_topics(articles['clean_title'].values[i])
                class_nums.append(output[0])
                class_probs.append(output[1])
                
            articles['topic_num'] = class_nums
            articles['topic_prob'] = class_probs
            
            id_list = []
            topic_list = []
            createdAt_list = []
            for article_id in articles['_id']:
                if article_id in article_dict.keys():
                    article = article_dict[article_id]
                    id_list.append(article['id'])
                    topic_list.append(article['topics'])
                    createdAt_list.append(article['createdAt'])
                else:
                    id_list.append([])
                    topic_list.append([])
                    createdAt_list.append([])

            articles['comment_id'] = id_list
            articles['comment_topics'] = topic_list
            articles['comment_createdAt'] = createdAt_list

            articles_by_day[current_date.strftime("%Y-%m-%d")] = articles
            current_date += relativedelta(days=1)
            del id_list, topic_list, createdAt_list, articles
    else:
        print('skip')
        current_date += relativedelta(days=1)

In [ ]:
threshold = 10
for day in articles_by_day.keys():
    articles = articles_by_day[day]
    print(day, len(articles[articles['comment_id'].apply(len) > threshold]))

In [ ]:
# pop element from articles_by_day if there are no articles
for day in list(articles_by_day.keys()):
    if len(articles_by_day[day]) == 0:
        print('pop', day)
        articles_by_day.pop(day)

# save articles_by_day to pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
    pickle.dump(articles_by_day, f)

## 2A. add comment embeddings to articles_by_day (full / AT, GWP, MJ)

In [ ]:
# load articles_by_day from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'rb') as f:
    articles_by_day = pickle.load(f)

In [ ]:
# remove articles with less than threshold comments
for day, articles in articles_by_day.items():
    articles_by_day[day] = articles[articles['comment_id'].apply(len) > threshold]

In [ ]:
date_list = [DATE_RANGES[collection_name][0].strftime('%m%y')]
while True:
    next_date = next_month(date_list[-1])
    if next_date == DATE_RANGES[collection_name][1].strftime('%m%y'):
        break
    else:
        date_list.append(next_date)

In [ ]:
embeddings_dict = {}
    
for date in date_list:
    print(date)
    next_date = next_month(date)
    embedding_foler = '/data/comments/valentin/sentence-embeddings/'
    embedding_path = embedding_foler + f'{collection_name.lower()}/bert-emb-{date}-{next_date}.pt'
    embedding = torch.load(embedding_path, map_location=torch.device('cpu'))
    if embedding['embeddings'].device.type == 'cuda':
        print(date, 'cuda')
        embedding['embeddings']= embedding['embeddings'].to('cpu')
    embeddings_dict[date] = embedding

In [ ]:
# full data 

for article_day in articles_by_day.keys():
    print(article_day)
    articles = articles_by_day[article_day]
    # assuming comment is sorted by createdAt
    comments_embeddings_list = []
    
    # if articles has column 'comments_embedding', then skip
    if 'comments_embeddings' in articles.columns:
        print(article_day, 'skip')
        continue
    
    for article in articles.itertuples():
        my_createdAt = [date.strftime('%m%y') for date in article.comment_createdAt]
        my_createdAt_by_month = defaultdict(list)
        start = 0
        for i, date in enumerate(my_createdAt):
            if i == len(my_createdAt) - 1 or date != my_createdAt[i+1]:
                my_createdAt_by_month[date].extend(list(range(start, i+1)))
                start = i+1
        my_createdAt_by_month = dict(my_createdAt_by_month)

        embeddings_month_article_list = []
        for comment_month in my_createdAt_by_month.keys():
            embeddings_month = embeddings_dict[comment_month]
            id_list = [article.comment_id[i] for i in my_createdAt_by_month[comment_month]]
            embeddings_month_article_list.append(np.array([embeddings_month['embeddings'][embeddings_month['_id'].index(comment_id)].numpy() for comment_id in id_list]))
        comments_embeddings_list.append(np.vstack(embeddings_month_article_list))
        
    articles_by_day[article_day] = articles_by_day[article_day].assign(comments_embeddings=comments_embeddings_list)
    
# save articles_by_month
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
    pickle.dump(articles_by_day, f)

In [ ]:
articles_by_day['2018-02-01']

## 2B. add comment embeddings to articles_by_day (30 days, / BB, TH)

In [ ]:
# load articles_by_day from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'rb') as f:
    articles_by_day = pickle.load(f)

In [ ]:
# remove articles with less than threshold comments
for day, articles in articles_by_day.items():
    articles_by_day[day] = articles[articles['comment_id'].apply(len) > threshold]

In [ ]:
# maximum 30 days: for preprocessing

date_cutoff = 30
topic_by_month = defaultdict(lambda: np.zeros(num_topics+1))
org_list = []
fil_list = []

for article_day in articles_by_day.keys():
    articles = articles_by_day[article_day]
    comment_createdAt_list = []
    comment_id_list = []
    comment_topics_list = []
    comments_embeddings_list = []
    topic_freq = Counter({i: 0 for i in range(-1, num_topics)})
    
    for article in articles.itertuples():
        filtered_index = np.where(np.array([(comment_createdAt - article.createdAt).days for comment_createdAt in article.comment_createdAt]) < date_cutoff)[0]
        comment_createdAt_list.append([article.comment_createdAt[i] for i in filtered_index])
        comment_id_list.append([article.comment_id[i] for i in filtered_index])
        comment_topics = [article.comment_topics[i] for i in filtered_index]
        comment_topics_list.append(comment_topics)
        topic_freq.update(comment_topics)
        
    # sort topic_freq by key and get values
    topic_freq = [topic_freq[key] for key in sorted(topic_freq.keys())]
    topic_by_month[article_day] = topic_freq
    
    articles_by_day[article_day] = articles_by_day[article_day].assign(comment_id=comment_id_list, comment_topics=comment_topics_list, comment_createdAt=comment_createdAt_list)

In [ ]:
def load_embeddings(date):
    print(date)
    next_date = next_month(date)
    embedding_foler = '/data/comments/valentin/sentence-embeddings/'
    embedding_path = embedding_foler + f'{collection_name.lower()}/bert-emb-{date}-{next_date}.pt'
    embedding = torch.load(embedding_path, map_location=torch.device('cpu'))
    if embedding['embeddings'].device.type == 'cuda':
        print(date, 'cuda')
        embedding['embeddings']= embedding['embeddings'].to('cpu')
    return embedding

In [ ]:
# max 2-month comments

for article_day in articles_by_day.keys():
    break

first=True
current_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
next_date = next_month(current_date)


for article_day in articles_by_day.keys():
    print(article_day)
    articles = articles_by_day[article_day]
    
    # if articles has column 'comments_embedding', then skip
    if 'comments_embeddings' in articles.columns:
        print(article_day, 'skip')
        current_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
        next_date = next_month(current_date)
        continue
    
    if first:
        current_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
        next_date = next_month(current_date)
        current_embeddings = load_embeddings(current_date)
        next_embeddings = load_embeddings(next_month(current_date)) 
        first = False
    
    comments_embeddings_list = []
    article_date = datetime.strptime(article_day, '%Y-%m-%d').strftime('%m%y')
    if current_date != article_date:  # next month happened)
        current_date = article_date
        next_date = next_month(current_date)
        current_embeddings = next_embeddings
        next_embeddings = load_embeddings(next_date)
        
        # save articles_by_month
        with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
            pickle.dump(articles_by_day, f)
    
    for article in articles.itertuples():
        
        embeddings_month_article_list = []
        my_createdAt = [date.strftime('%m%y') for date in article.comment_createdAt]
        my_createdAt_by_month = defaultdict(list)
        start = 0
        for i, date in enumerate(my_createdAt):
            if i == len(my_createdAt) - 1 or date != my_createdAt[i+1]:
                my_createdAt_by_month[date].extend(list(range(start, i+1)))
                start = i+1
        my_createdAt_by_month = dict(my_createdAt_by_month)
        
        if current_date in my_createdAt_by_month.keys():
            createdAt_current_month = my_createdAt_by_month[current_date]
            id_current_list = [article.comment_id[i] for i in createdAt_current_month]
            if len(id_current_list)>0:
                embeddings_month_article_list.append(np.array([current_embeddings['embeddings'][current_embeddings['_id'].index(comment_id)].numpy() for comment_id in id_current_list]))
        
        
        if next_date in my_createdAt_by_month.keys():
            createdAt_next_month = my_createdAt_by_month[next_date]
            id_next_list = [article.comment_id[i] for i in createdAt_next_month]
            if len(id_next_list)>0:
                embeddings_month_article_list.append(np.array([next_embeddings['embeddings'][next_embeddings['_id'].index(comment_id)].numpy() for comment_id in id_next_list]))
        
        if len(embeddings_month_article_list) > 0:
            comments_embeddings_list.append(np.vstack(embeddings_month_article_list))
        else:
            comments_embeddings_list.append(np.array([]))
        
    articles_by_day[article_day] = articles_by_day[article_day].assign(comments_embeddings=comments_embeddings_list)
    
# save articles_by_month
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
    pickle.dump(articles_by_day, f)

In [ ]:
# save articles_by_month
with open(join('article', collection_name.lower(), model_name, 'articles_by_day.pkl'), 'wb') as f:
    pickle.dump(articles_by_day, f)

## 3. total_embedding_list / total_nonexist_index_list

In [ ]:
total_mean_embedding_list= []
total_nonexist_index_list = []
for article_day in articles_by_day.keys():
    print(article_day)
    articles = articles_by_day[article_day]
    embedding_list = [[] for _ in range(num_topics+1)] # including -1
    mean_embedding_list = [[] for _ in range(num_topics+1)] # including -1
    nonexist_index_list = [] 
    for article in articles.itertuples():
        for i in range(num_topics+1):  # including -1  
            comment_index = np.where(np.array(article.comment_topics) == i-1)[0]
            if len(comment_index)>0:
                embedding_list[i].append(article.comments_embeddings[comment_index])
            
    for i in range(num_topics+1): 
        if len(embedding_list[i])>0:
            averaged = np.vstack(embedding_list[i]).mean(axis=0)
        else:
            nonexist_index_list.append(i)
            averaged = np.zeros(384)  # embedding shape
        mean_embedding_list[i] = averaged

    print(nonexist_index_list)
    mean_embedding_list = np.array(mean_embedding_list)
    total_mean_embedding_list.append(mean_embedding_list)
    total_nonexist_index_list.append(nonexist_index_list)
    
total_mean_embedding_list = np.array(total_mean_embedding_list)
# save total_mean_embedding_list and total_nonexist_index_list as pickle
with open(join('article', collection_name.lower(), model_name, 'total_mean_embedding_day_list.pkl'), 'wb') as f:
    pickle.dump(total_mean_embedding_list, f)
with open(join('article', collection_name.lower(), model_name, 'total_nonexist_index_day_list.pkl'), 'wb') as f:
    pickle.dump(total_nonexist_index_list, f)

In [ ]:
# for each element of total_mean_embedding_list, remove all-zero rows (non-existing topics) and concatenate all of them.
embedding_list_tsne = []

for i, mean_embedding in enumerate(total_mean_embedding_list):
    mean_embedding = np.delete(mean_embedding, total_nonexist_index_list[i], axis=0)
    embedding_list_tsne.append(mean_embedding)
    
embedding_list_tsne = np.concatenate(embedding_list_tsne, axis=0)
X_embedded = sk.manifold.TSNE(n_components=2).fit_transform(embedding_list_tsne)

total_tsne_list = []
previous_list = np.array([[0, 0]] * (num_topics+1)) # including -1
counter = 0
for i in range(len(articles_by_day.keys())):
    print(i)
    tsne_list = []
    nonexist_index_list = deepcopy(total_nonexist_index_list[i])
    if len(nonexist_index_list)==0:
        tsne_list = X_embedded[counter:counter+num_topics+1]
        previous_list = X_embedded[counter:counter+num_topics+1]
        counter+=num_topics+1
    else:
        current_index = nonexist_index_list[0]
        for j in range(num_topics+1):
            if j==current_index:
                tsne_list.append(previous_list[j])
                nonexist_index_list.pop(0)
                if len(nonexist_index_list)>0:
                    current_index = nonexist_index_list[0]
            else:
                tsne_list.append(X_embedded[counter])
                previous_list[j] = X_embedded[counter]
                counter+=1
            
    assert len(tsne_list)==num_topics+1
    total_tsne_list.append(np.array(tsne_list))
    
total_tsne_list = np.array(total_tsne_list)

with open(join('article', collection_name.lower(), model_name, f'total_tsne_list.pkl'), 'wb') as f:
    pickle.dump(total_tsne_list, f)

## 4. Title / comments similarity ratio_dict

In [ ]:
topic_tuple_list_dict = defaultdict(list)
for day in articles_by_day.keys():
    articles = articles_by_day[day]
    for article in articles.itertuples():
        topic_tuple = tuple(sorted(article.topic_num[:3]))
        topic_tuple_list_dict[topic_tuple].append((article._1, article.createdAt.strftime('%Y-%m-%d')))
        
# get the number of elements in each key
num_elements = {key: len(value) for key, value in topic_tuple_list_dict.items()}

# sort the dictionary by the length of the lists
from collections import OrderedDict
sorted_dict = OrderedDict(sorted(num_elements.items(), key=lambda item: item[1], reverse=True))

In [ ]:
topic_tuple_list_dict[(52, 111, 213)]

In [ ]:
# control : monthly article based

from tqdm import tqdm

batch_size = 10000
ratio_dict = {}

for key in tqdm(list(sorted_dict.keys())):
    if -1 in key:
        print(key, 'skip')
        continue
    comments_embeddings_dict_self = {key[i] : [] for i in range(len(key))}
    comments_embeddings_dict_others = {key[i] : [] for i in range(len(key))}

    for month in sorted(list(set([t[1] for t in topic_tuple_list_dict[key]]))):
        articles = articles_by_day[day]
        for article in articles.itertuples():
            topic_tuple = tuple(sorted(article.topic_num[:3]))
            if topic_tuple == key:
                for k in key:
                    comment_indices = np.where(np.isin(article.comment_topics, k))[0]
                    if len(comment_indices):
                        comments_embeddings_dict_self[k].append(article.comments_embeddings[comment_indices])
            else:
                for k in key:
                    comment_indices = np.where(np.isin(article.comment_topics, k))[0]
                    if len(comment_indices):
                        comments_embeddings_dict_others[k].append(article.comments_embeddings[comment_indices])
                
    comparable=True
    
    for k in key:
        if len(comments_embeddings_dict_self[k]) == 0 or len(comments_embeddings_dict_others[k]) == 0:
            comparable = False
            continue
        comments_embeddings_dict_self[k] = np.vstack(comments_embeddings_dict_self[k])
        comments_embeddings_dict_others[k] = np.vstack(comments_embeddings_dict_others[k])
    
    if comparable:
        pair_list = list(combinations(key, 2))
        
        test_cos_sim_matrix_self = [cosine_similarity(comments_embeddings_dict_self[i], comments_embeddings_dict_self[j]) for i, j in pair_list] 
        test_sim_list_self = [np.mean(matrix) for matrix in test_cos_sim_matrix_self]
        del test_cos_sim_matrix_self
        
        test_sim_list_others = []
        for i, j in pair_list:
            mean_list = []
            size_list = []
            for k in range(0, len(comments_embeddings_dict_others[i]), batch_size):
                batch_i = comments_embeddings_dict_others[i][k:k+batch_size]
                mean_list.append(np.mean(cosine_similarity(batch_i, comments_embeddings_dict_others[j])))
                size_list.append(len(batch_i))
            
            # get true mean
            test_sim_list_others.append(np.average(mean_list, weights=size_list))

        ratio_dict[key] = [test_sim_list_self[i] / test_sim_list_others[i] for i in range(len(test_sim_list_self))]
        
    del comments_embeddings_dict_self, comments_embeddings_dict_others
            
if len(ratio_dict):
    average_ratio = np.mean(list(ratio_dict.values()), axis=0)
    print(average_ratio)
else:
    print(key, 'not comparable')
    
# save ratio_dict
with open(join('article', collection_name.lower(), model_name, 'ratio_dict.pkl'), 'wb') as f:
    pickle.dump(ratio_dict, f)